#### Retrieve first 10 fish products from the Vons grocery store website (Save to a dataframe)

In [1]:
from urllib.parse import urlparse
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np

url = 'https://www.vons.com/shop/aisles/meat-seafood/fish-shellfish/fish.html?sort=&page=1&loc=2053'
soup = BeautifulSoup(requests.get(url).text, 'html.parser')

df_products=pd.DataFrame(columns=['Product_id', 'Url', 'Description'])

# Use this to get the url netloc
host = urlparse(url).netloc
scheme = urlparse(url).scheme

# Get href links of products in the fish and seafood department
fish_links = soup.find_all('a', class_='product-title__name')

if fish_links:
    for i, fish_link in enumerate(fish_links[:10]): # only run for the first 5 rows

        # Add the url to the list
        href = fish_link.get("href")
        if href:
            url = scheme+'://'+host+href
        else: url = None

        # Add the product_id to the list
        prod_id = fish_link.get("id")
        if not prod_id:
            prod_id = None   

        # Add the description to the list
        prod_text = fish_link.get_text()
        if not prod_text:
            prod_text = None

        # Insert new row into the products dataframe
        df_products.loc[len(df_products)]= [prod_id,
                                      url,
                                      prod_text]

# Show the 10 products retrieved from the website in dataframe form
df_products

,Product_id,Url,Description
0,pg186190041,https://www.vons.com/shop/product-details.1861...,Fresh Farmed Atlantic Salmon Fillet Color Adde...
1,pg970033271,https://www.vons.com/shop/product-details.9700...,Atlantic Salmon Portion - Minimum 8 oz - ea\n ...
2,pg960264310,https://www.vons.com/shop/product-details.9602...,waterfront BISTRO Tilapia Fillets Bonesless & ...
3,pg960227381,https://www.vons.com/shop/product-details.9602...,waterfront BISTRO Boneless Skinless Wild Alask...
4,pg960190258,https://www.vons.com/shop/product-details.9601...,waterfront BISTRO Boneless Skin On Wild Alaska...
5,pg960137751,https://www.vons.com/shop/product-details.9601...,Fresh Atlantic Salmon Color Added Portion 1 ct...
6,pg960139116,https://www.vons.com/shop/product-details.9601...,Atlantic Salmon Portion 5 Oz Fresh 1 Count - E...
7,pg960122401,https://www.vons.com/shop/product-details.9601...,Open Nature Skin On Wild Caught Alaskan Sockey...
8,pg970027909,https://www.vons.com/shop/product-details.9700...,Signature SELECT Pacific Cod Fillet Raw Previo...
9,pg960035112,https://www.vons.com/shop/product-details.9600...,waterfront BISTRO Boneless Skin On Wild Alaska...


#### Retrieve first 5 fish products from the Vons grocery store website (Save to a dictionary)

In [8]:
from urllib.parse import urlparse
from bs4 import BeautifulSoup
import requests

url = 'https://www.vons.com/shop/aisles/meat-seafood/fish-shellfish/fish.html?sort=&page=1&loc=2053'
soup = BeautifulSoup(requests.get(url).text, 'html.parser')

product={}

# Use this to get the url netloc
host = urlparse(url).netloc
scheme = urlparse(url).scheme

# Get href links of products in the fish and seafood department
fish_links = soup.find_all('a', class_='product-title__name')

if fish_links:
    for i, fish_link in enumerate(fish_links[:5]): # only run for the first 5 rows
        #print(fish_link)
        href = fish_link.get("href")
        if href:
            # Add the url to the dictionary
            #key_id = 'url'+str(x)
            url = scheme+'://'+host+href
            product[i] = [url]

            # Add the product_id to the dictionary
            prod_id = fish_link.get("id")
            product[i].append(prod_id)

            # Add the description to the dictionary
            prod_text = fish_link.get_text()
            product[i].append(prod_text.strip())

# Show the 5 products retrieved from the website in dictionary form
for key, value in product.items():
    print(value)
        

['https://www.vons.com/shop/product-details.186190041.html', 'pg186190041', 'Fresh Farmed Atlantic Salmon Fillet Color Added  - 1.5 lb']
['https://www.vons.com/shop/product-details.970033271.html', 'pg970033271', 'Atlantic Salmon Portion - Minimum 8 oz - ea']
['https://www.vons.com/shop/product-details.960264310.html', 'pg960264310', 'waterfront BISTRO Tilapia Fillets Bonesless & Skinless - 16 Oz']
['https://www.vons.com/shop/product-details.960227381.html', 'pg960227381', 'waterfront BISTRO Boneless Skinless Wild Alaskan Cod Fillets - 16 Oz']
['https://www.vons.com/shop/product-details.960190258.html', 'pg960190258', 'waterfront BISTRO Boneless Skin On Wild Alaskan Pink Salmon Fillets - 16 Oz']


#### CSV Export

In [4]:
import csv

# Write products from www.vons.com into a csv file
column_names = ['URL','Product_Id', 'Description']
with open('vons.csv','w') as outfile:
    spamwriter = csv.writer(outfile, lineterminator='\n')
    spamwriter.writerow(column_names)
    for dict_value in product.values():
        spamwriter.writerow(dict_value)

#### JSON Export

In [5]:
import json

with open('vons.json','w') as outfile:
    json.dump(product, outfile)

#### SQLite3 Database Export

In [7]:
# Export Von grocery list to SQLite3 database
import sqlite3

table_ddl = """
CREATE TABLE IF NOT EXISTS vons(
    product_id TEXT PRIMARY KEY,
    url TEXT,
    description TEXT
)
"""

sqlite_insert = """
INSERT OR REPLACE INTO vons
    values(?, ?, ?)
"""

def save_to_sqlite(database_path, dataframe):
    global connection
    connection = __connect(database_path)

    # Check to verify db connection
    if not connection: return

    # Check to verify table exists
    try: 
        __ensure_table(connection)
    except sqlite3.Error as e:
        print(f"SQLite error: {e}")
        __close_connection()
        
    # Save data to table
    try:
        for row in range(len(dataframe)):
            __save_row(row, dataframe)
    except sqlite3.Error as e:
        print(f"SQLite error: {e}")
        __close_connection()
    
    __close_connection()

def __connect(database):
    try:
        conn = sqlite3.connect(database)
        cursor = conn.cursor()
        # Attempt simple query to verify connection
        cursor.execute("SELECT 1")
        print(f'Database connection established')
        return sqlite3.connect(database)
    except sqlite3.Error as e:
        print(f'Database connection error: {e}')
        return False

def __close_connection():
    
    if connection:
        connection.close()
        print(f'Database connection is closed')

def __ensure_table(conn):

    cursor = conn.cursor()
    cursor.execute(f"SELECT Product_id FROM vons LIMIT 1")
    result = cursor.fetchone()
    if not result:
        print(f'Creating Vons table')
        connection.execute(table_ddl)
    else: print(f'Table already exists')
    
def __save_row(row, dataframe):
    connection.execute(sqlite_insert, (
        dataframe.iloc[row]['Product_id'],
        dataframe.iloc[row]['Url'],
        dataframe.iloc[row]['Description']))
    connection.commit()
    print(f'Row {row+1} has been saved')

# Run the sql to save records to a table
# Create table if it doesn't exist
save_to_sqlite('my_database.db', df_products)

Database connection established
Table already exists
Row 1 has been saved
Row 2 has been saved
Row 3 has been saved
Row 4 has been saved
Row 5 has been saved
Row 6 has been saved
Row 7 has been saved
Row 8 has been saved
Row 9 has been saved
Row 10 has been saved
Database connection is closed
